In [0]:
# source
ECOMM_BRONZE_FILES = "abfss://bronze@gopaldatalake.dfs.core.windows.net/ecomm-gopal/"
# target
ECOMM_SILVER_FILES = "abfss://silver@gopaldatalake.dfs.core.windows.net/ecomm-gopal/"

In [0]:
# read from gksdatalake2, container bronze, ecomm from adls, csv files
# testing purpose only, not used
# BATCH SPARK.READ
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(ECOMM_BRONZE_FILES)

df.printSchema()
df.show(2)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# Step 1: Define explicit schema
ecomm_schema = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", StringType(), True), # the csv does not have sql datetime format
    StructField("UnitPrice", DoubleType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("Country", StringType(), True)
])

# LAZY , not yet started the stream
# Step 1: Read from Bronze Zone
bronze_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(ecomm_schema)
    .load(ECOMM_BRONZE_FILES)  
)

# .show will not work on stream
bronze_df.printSchema()
bronze_df.show(2)

In [0]:
# data partitioning
# write the ecomm data into silver, by country wise partition
# parquet format, this is not delta table

(bronze_df.write
 .mode("overwrite")
 .partitionBy("Country")
 .format("parquet")
 .save(ECOMM_SILVER_FILES))
# read the parquet file from silver zone
# bronze_df = spark.read.parquet(ECOMM_SILVER


In [0]:
(bronze_df.coalesce(1).write
 .mode("overwrite")
 .format("parquet")
 .save("abfss://silver@gopaldatalake.dfs.core.windows.net/ecomm-single-file/"))

In [0]:
# advantage of partition
# df.partitionBy("Country", "State", "District")
"""
India -   1000 GB 
  KA - 100 GB 
    BLR district - 10 GB 
      file1.par
      file2.par
      
    Mys          - 2 GB
         file200.par

  TN - 125 GB 
    CHN district - 10 GB 
     
USA 
"""

"""
Assume we have not done partition by country and state,

your query over non-parttiioned dataset
data - 1000 GB
  file1.par
  ..
  file100000.par

SELECT ... group by ....   - READ 1000 GB, process 1000 GB  over non-parttiioned 
SELECT ... group by .... where country = India and State = 'KA'  - over non-parttiioned  READ 1000 GB, process 100 GB 

"""

"""assume you do query over partition data

SELECT ... group by ....  - READ 1000 GB, process 1000 GB  over  parttiioned data as well, no fitler for partition columns

SELECT ... group by .... where country = India and State = 'KA'  - over parttiioned  
 it reads from Country=India sub directory and State=KA sub directory, skip other files

   read 100 GB, process 100 GB , avoid reading other countrries and other states data

SELECT ... group by .... where country = India and State = 'KA' and District=BLR - over parttiioned  
    read  10 GB , process 10 GB
"""

"""
dept=Sales
    Year=2025
      Month=01
        Day=01
           file1.par
"""

In [0]:
partitionedDf = spark.read.parquet(ECOMM_SILVER_FILES)
partitionedDf.printSchema()
partitionedDf.show(2)
partitionedDf.explain()
# read the parquet file from silver zone

In [0]:
df2 = partitionedDf.filter("Country = 'United Kingdom' ")
df2.explain()

In [0]:
# 5 mins 
# now, convert the InvoiceDate date from string to Timestamp, 
# extract year, month, and day
# write to silver layer comm-daywise folder 
#    Year=2025
#      Month=01
#        Day=01
#           file1.par

In [0]:
# conver time stamp from 12/1/2010 8:26 to timestamp
import pyspark.sql.functions as F 
df = bronze_df.withColumn("InvoiceDateTime", F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm"))\
              .withColumn("year", F.year("InvoiceDateTime"))\
                  .withColumn("month", F.month("InvoiceDateTime"))\
                  .withColumn("day", F.dayofmonth("InvoiceDateTime"))

ECOMM_SILVER_DAYWISE_FILES = "abfss://silver@gopaldatalake.dfs.core.windows.net/ecomm-daywise-gopal/"
df.write.mode("overwrite").partitionBy("year", "month", "day").format("parquet").save(ECOMM_SILVER_DAYWISE_FILES)
# df.printSchema()
# df.show(2)